# XGBoost / LightGBM：梯度提升树

本 notebook 使用 `predictive_maintenance.csv` 演示两种常用的梯度提升树模型：

- **XGBoost**
- **LightGBM**

主要内容包括：

1. 梯度提升树的基本思想；
2. 数据拆分、类别特征编码；
3. XGBoost 与 LightGBM 训练；
4. 使用验证集早停，避免树数量过多导致过拟合；
5. 评估混淆矩阵、Precision、Recall、F1；
6. 查看特征重要度；
7. 使用类别不平衡处理与阈值调整。

## 数据集说明

目标变量：

- `Machine failure`：`1` 表示设备故障，`0` 表示正常。

输入特征：

- `Type`：设备类型；
- `Air temperature`：空气温度；
- `Process temperature`：过程温度；
- `Rotational speed`：转速；
- `Torque`：扭矩；
- `Tool wear`：刀具磨损。

`TWF`、`HDF`、`PWF`、`OSF`、`RNF` 是故障原因/故障模式指示变量，直接使用会造成标签泄露，因此不作为输入特征。

## 1. 梯度提升树的核心思想

梯度提升树（Gradient Boosted Decision Trees, GBDT）是一种集成学习方法。它不是只训练一棵树，而是连续训练多棵决策树：

1. 先训练一棵基础树；
2. 计算当前模型的预测误差；
3. 下一棵树重点学习这些误差或损失函数的梯度方向；
4. 把所有树的输出按照学习率累加起来，得到最终预测。

因此，每一棵新树都在“修正前面树的错误”。

### 常见参数

- `n_estimators`：树的数量；
- `learning_rate`：每棵树的贡献缩放比例；
- `max_depth`：单棵树的最大深度；
- `num_leaves`：LightGBM 中叶节点数量；
- `subsample`：每棵树使用的样本比例；
- `colsample_bytree`：每棵树使用的特征比例；
- `early_stopping`：验证集指标不再提升时提前停止训练。

一般来说，`learning_rate` 越小，通常需要更多树；树太深或叶子太多，更容易过拟合。

## 2. 导入库

In [ ]:
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.compose import ColumnTransformer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

## 3. 读取数据

In [ ]:
csv_path = Path.cwd() / "predictive_maintenance.csv"

if not csv_path.exists():
    csv_path = Path("Python Guidance/day31-45/predictive_maintenance.csv")

df = pd.read_csv(csv_path)
df.head()

## 4. 定义特征和目标变量

In [ ]:
target = "Machine failure"
failure_mode_columns = ["TWF", "HDF", "PWF", "OSF", "RNF"]

numeric_features = [
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear",
]
categorical_features = ["Type"]

X = df[numeric_features + categorical_features]
y = df[target]

## 5. 检查目标变量分布

该数据集的故障样本较少，属于类别不平衡问题。因此评估模型时，除了 Accuracy，还要重点关注故障类别的 Precision、Recall 和 F1。

In [ ]:
print("数据形状:", df.shape)
print("缺失值数量:", int(df.isna().sum().sum()))
print("\n目标变量分布:")
print(y.value_counts().rename(index={0: "正常", 1: "故障"}))
print(f"\n故障比例: {y.mean():.2%}")

In [ ]:
df[numeric_features].describe().T

## 6. 拆分训练集、验证集和测试集

梯度提升树通常使用验证集做早停。因此这里将数据拆成三部分：

- 训练集：拟合模型；
- 验证集：选择早停轮数、调节参数；
- 测试集：最终评估。

最终比例约为 `60% / 20% / 20%`。

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    random_state=42,
    stratify=y_train_val,
)

print("训练集:", X_train.shape, f"故障比例 {y_train.mean():.2%}")
print("验证集:", X_val.shape, f"故障比例 {y_val.mean():.2%}")
print("测试集:", X_test.shape, f"故障比例 {y_test.mean():.2%}")

## 7. 类别特征编码

树模型不需要像 KNN/SVM 那样对数值特征做标准化，但 `Type` 仍然是类别特征，需要先转换成数值形式。

这里继续使用 One-Hot 编码，避免人为引入 `L < M < H` 的顺序关系。

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc = preprocessor.transform(X_val)
X_test_enc = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

print("编码后训练集:", X_train_enc.shape)
print("编码后验证集:", X_val_enc.shape)
print("编码后测试集:", X_test_enc.shape)
print("特征名:", list(feature_names))

## 8. 定义评估函数

In [ ]:
def metrics_at_threshold(y_true, probability, threshold=0.5):
    y_pred = (probability >= threshold).astype(int)
    y_true_array = y_true.to_numpy()

    tp = int(((y_pred == 1) & (y_true_array == 1)).sum())
    tn = int(((y_pred == 0) & (y_true_array == 0)).sum())
    fp = int(((y_pred == 1) & (y_true_array == 0)).sum())
    fn = int(((y_pred == 0) & (y_true_array == 1)).sum())

    return {
        "threshold": threshold,
        "accuracy": float((y_pred == y_true_array).mean()),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "false_positives": fp,
        "false_negatives": fn,
    }


def evaluate_model(name, model, X_eval, y_eval, threshold=0.5):
    probability = model.predict_proba(X_eval)[:, 1]
    row = metrics_at_threshold(y_eval, probability, threshold)
    row["model"] = name
    return row


def print_model_report(name, model):
    y_pred = model.predict(X_test_enc)
    print(name)
    print(classification_report(y_test, y_pred, target_names=["正常", "故障"], digits=3))

## 9. 训练 XGBoost

XGBoost 是梯度提升树的一种高效实现，加入了正则化、列采样、稀疏数据处理等优化。

这里使用验证集进行早停：如果验证集 AUC 连续多轮没有提升，就停止训练。

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30,
)

xgb_model.fit(
    X_train_enc,
    y_train,
    eval_set=[(X_val_enc, y_val)],
    verbose=False,
)

print("XGBoost 早停后的最佳轮数:", xgb_model.best_iteration)

In [ ]:
print_model_report("XGBoost 测试集结果", xgb_model)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    xgb_model.predict(X_test_enc),
    display_labels=["正常", "故障"],
    cmap="Blues",
    values_format="d",
)
plt.title("XGBoost 混淆矩阵")
plt.show()

## 10. 训练 LightGBM

LightGBM 也是梯度提升树实现，特点是训练速度快、内存占用低。它通常使用 leaf-wise 树生长策略，并支持类别特征和高效直方图算法。

这里同样使用验证集早停。

In [ ]:
lgbm_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary",
    importance_type="gain",
    random_state=42,
    n_jobs=-1,
    force_col_wise=True,
    verbosity=-1,
)

lgbm_model.fit(
    X_train_enc,
    y_train,
    eval_set=[(X_val_enc, y_val)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(stopping_rounds=30, verbose=False),
        lgb.log_evaluation(period=0),
    ],
)

print("LightGBM 早停后的最佳轮数:", lgbm_model.best_iteration_)

In [ ]:
print_model_report("LightGBM 测试集结果", lgbm_model)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    lgbm_model.predict(X_test_enc),
    display_labels=["正常", "故障"],
    cmap="Blues",
    values_format="d",
)
plt.title("LightGBM 混淆矩阵")
plt.show()

## 11. 对比 XGBoost 和 LightGBM

In [ ]:
baseline_results = pd.DataFrame(
    [
        evaluate_model("XGBoost", xgb_model, X_test_enc, y_test),
        evaluate_model("LightGBM", lgbm_model, X_test_enc, y_test),
    ]
)

baseline_results[[
    "model",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "false_positives",
    "false_negatives",
]]

## 12. 查看特征重要度

特征重要度可以帮助我们理解模型主要依赖哪些变量进行判断。但重要度不等于因果关系，只能说明模型在该数据训练过程中更常使用或获得更大收益。

In [ ]:
def plot_top_features(importances, title, ax):
    importance = pd.Series(importances, index=feature_names)
    importance = importance.sort_values(ascending=False).head(10)

    ax.barh(importance.index[::-1], importance.values[::-1])
    ax.set_title(title)
    ax.set_xlabel("Importance")
    ax.grid(True, axis="x", alpha=0.3)


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_top_features(xgb_model.feature_importances_, "XGBoost 特征重要度", axes[0])
plot_top_features(lgbm_model.feature_importances_, "LightGBM 特征重要度", axes[1])
plt.tight_layout()
plt.show()

## 13. 处理类别不平衡

故障样本只占约 3.4%。普通模型可能倾向于把样本预测为“正常”，从而提高 Accuracy，但会漏掉真实故障。

常见处理方式包括：

- XGBoost：使用 `scale_pos_weight` 提高少数类误分类代价；
- LightGBM：使用 `class_weight="balanced"`；
- 采样方法：过采样少数类或欠采样多数类；
- 阈值调整：降低故障预测阈值，换取更高 Recall。

下面先比较两个库自带的类别权重参数。

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight}")

xgb_balanced = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30,
)

xgb_balanced.fit(
    X_train_enc,
    y_train,
    eval_set=[(X_val_enc, y_val)],
    verbose=False,
)

lgbm_balanced = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary",
    class_weight="balanced",
    importance_type="gain",
    random_state=42,
    n_jobs=-1,
    force_col_wise=True,
    verbosity=-1,
)

lgbm_balanced.fit(
    X_train_enc,
    y_train,
    eval_set=[(X_val_enc, y_val)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(stopping_rounds=30, verbose=False),
        lgb.log_evaluation(period=0),
    ],
)

print("XGBoost balanced 最佳轮数:", xgb_balanced.best_iteration)
print("LightGBM balanced 最佳轮数:", lgbm_balanced.best_iteration_)

In [ ]:
all_results = pd.DataFrame(
    [
        evaluate_model("XGBoost", xgb_model, X_test_enc, y_test),
        evaluate_model("XGBoost balanced", xgb_balanced, X_test_enc, y_test),
        evaluate_model("LightGBM", lgbm_model, X_test_enc, y_test),
        evaluate_model("LightGBM balanced", lgbm_balanced, X_test_enc, y_test),
    ]
)

all_results[[
    "model",
    "accuracy",
    "precision",
    "recall",
    "f1",
    "false_positives",
    "false_negatives",
]]

## 14. 调整阈值：继续权衡漏报和误报

类别权重会直接改变模型训练目标；阈值调整则是在模型训练完成后，移动“判定为故障”的边界。

- 降低阈值：Recall 通常上升，FP 通常增加；
- 提高阈值：FP 通常减少，但 FN 可能增加。

In [ ]:
rows = []

for model_name, model in [
    ("XGBoost balanced", xgb_balanced),
    ("LightGBM balanced", lgbm_balanced),
]:
    probability = model.predict_proba(X_test_enc)[:, 1]
    for threshold in [0.3, 0.5, 0.7, 0.8]:
        row = metrics_at_threshold(y_test, probability, threshold)
        row["model"] = model_name
        rows.append(row)

threshold_results = pd.DataFrame(rows)[[
    "model",
    "threshold",
    "precision",
    "recall",
    "f1",
    "false_positives",
    "false_negatives",
]]

threshold_results

In [ ]:
plt.figure(figsize=(9, 5))

for model_name in threshold_results["model"].unique():
    subset = threshold_results[threshold_results["model"] == model_name]
    plt.plot(
        subset["threshold"],
        subset["recall"],
        marker="o",
        label=f"{model_name} Recall",
    )
    plt.plot(
        subset["threshold"],
        subset["precision"],
        marker="o",
        linestyle="--",
        label=f"{model_name} Precision",
    )

plt.xlabel("故障概率阈值")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.title("不同阈值下的 Precision / Recall")
plt.show()

## 15. 预测新样本

In [ ]:
new_sample = pd.DataFrame(
    [
        {
            "Type": "M",
            "Air temperature": 299.0,
            "Process temperature": 311.0,
            "Rotational speed": 1650,
            "Torque": 55.0,
            "Tool wear": 180,
        }
    ]
)

new_sample_enc = preprocessor.transform(new_sample)

for model_name, model in [
    ("XGBoost balanced", xgb_balanced),
    ("LightGBM balanced", lgbm_balanced),
]:
    probability = model.predict_proba(new_sample_enc)[0, 1]
    prediction = model.predict(new_sample_enc)[0]
    print(model_name)
    print("  预测结果:", "故障" if prediction == 1 else "正常")
    print(f"  故障概率: {probability:.3f}")

new_sample

## 16. 总结

本 notebook 演示了 XGBoost 和 LightGBM 两种梯度提升树模型：

- GBDT 通过逐棵树拟合前序模型的误差来完成集成学习；
- `learning_rate`、`n_estimators`、`max_depth`、`num_leaves` 等参数共同控制模型复杂度；
- 树模型不需要数值标准化，但类别特征仍需要编码；
- 验证集早停可以减少过拟合风险；
- 类别不平衡时，Accuracy 容易误导，应重点看故障类别的 Recall、Precision 和 F1；
- XGBoost 可以使用 `scale_pos_weight`，LightGBM 可以使用 `class_weight="balanced"`；
- 在模型训练后，也可以通过调整阈值在漏报和误报之间做业务权衡。

## 可以进一步尝试

1. 使用 `GridSearchCV` 或 `Optuna` 调参；
2. 尝试 SMOTE 等少数类过采样方法；
3. 比较随机森林、XGBoost、LightGBM 和 CatBoost；
4. 使用 SHAP 更细致地解释单个预测；
5. 根据设备维护成本，选择让总成本最低的分类阈值。